In [9]:
import pandas as pd
import numpy as np
import joblib

# ==============================
# LOAD
# ==============================

model = joblib.load("../models/regressor.pkl")
features = joblib.load("../models/features.pkl")

df = pd.read_csv("../data/final_dataset.csv")

X = df[features]

# ==============================
# PREDICT
# ==============================

df["ExpectedPrice"] = model.predict(X)

# ==============================
# ROUTE STATS (ROBUST FIX)
# ==============================

# FORCE CLEAN COPY (IMPORTANT)
df = df.copy()

# ensure correct types (safe)
df["Source"] = df["Source"].astype(int)
df["Destination"] = df["Destination"].astype(int)

route_stats = df.groupby(["Source", "Destination"], as_index=False)["Price"].mean()
route_stats.rename(columns={"Price": "price_avg"}, inplace=True)

df = df.merge(route_stats, on=["Source", "Destination"], how="left")

# HARD GUARANTEE
assert "price_avg" in df.columns, "price_avg missing after merge!"

df["price_avg"] = df["price_avg"].fillna(df["Price"].mean())

# ==============================
# METRICS
# ==============================

df["PriceDiff_%"] = (
    (df["ExpectedPrice"] - df["price_avg"]) / df["price_avg"]
) * 100

# ==============================
# DECISION ENGINE (REAL LOGIC)
# ==============================

def decision(row):
    if row["PriceDiff_%"] < -12:
        return "🔥 BUY NOW"
    elif row["PriceDiff_%"] < 0:
        return "BUY"
    elif row["PriceDiff_%"] < 15:
        return "WAIT"
    else:
        return "❌ OVERPRICED"

df["Decision"] = df.apply(decision, axis=1)

# ==============================
# RISK LEVEL
# ==============================

df["RiskLevel"] = pd.cut(
    df["PriceDiff_%"],
    bins=[-100, -12, 0, 15, 100],
    labels=["LOW", "MEDIUM", "HIGH", "OVERPRICED"]
)

# ==============================
# CLEAN SAVE
# ==============================

df.to_csv("../data/predictions_enriched.csv", index=False)

print("✅ Done")
print(df[["PriceDiff_%", "Decision"]].head())
print(df["Decision"].value_counts())

AssertionError: price_avg missing after merge!